In [ ]:
import time
import re
import shutil
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Bidirectional, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import transformers
from transformers import (
    MT5Tokenizer,
    MT5ForConditionalGeneration,
    AdamW
)
from torch import bfloat16

import nltk
from nltk.translate.gleu_score import sentence_gleu
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.metrics import edit_distance
import editdistance

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
# Tú impersonal

df = pd.read_csv('/content/tu_impersonal.csv')

In [ ]:
# Gerundio de posterioridad

# df = pd.read_csv('/content/gerundio.csv')

In [ ]:
# =============================
# 0. Preprocesamiento
# =============================

df['input'] = df['input'].str.lower()
df['target'] = df['target'].str.lower()


def eliminar_espacios_innecesarios(texto):
    texto = texto.strip()
    texto = re.sub(r'\s{2,}', ' ', texto)
    return texto

def añadir_punto_final(texto):
    texto = texto.strip()
    if not texto.endswith('.'):
        texto += '.'
    return texto

df['input'] = df['input'].apply(añadir_punto_final)
df['target'] = df['target'].apply(añadir_punto_final)
df['input'] = df['input'].apply(eliminar_espacios_innecesarios)
df['target'] = df['target'].apply(eliminar_espacios_innecesarios)

In [ ]:
# Gerundio de posterioridad

# df = df.rename(columns={
#     'Frase Incorrecta (Gerundio)': 'input',
#     'Frase Correcta': 'target'
# })

# df['input'] = df['input'].str.lower()
# df['target'] = df['target'].str.lower()


# def eliminar_espacios_innecesarios(texto):
#     texto = texto.strip()
#     texto = re.sub(r'\s{2,}', ' ', texto)
#     return texto

# def añadir_punto_final(texto):
#     texto = texto.strip()
#     if not texto.endswith('.'):
#         texto += '.'
#     return texto

# df['input'] = df['input'].apply(añadir_punto_final)
# df['target'] = df['target'].apply(añadir_punto_final)
# df['input'] = df['input'].apply(eliminar_espacios_innecesarios)
# df['target'] = df['target'].apply(eliminar_espacios_innecesarios)

In [ ]:
train_data, test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Entrenamiento: {train_data.shape}, Prueba: {test.shape}")

test_data, val_data = train_test_split(test, test_size=0.5, random_state=42)
print(f"Prueba: {test_data.shape}, Validación: {val_data.shape}")

In [ ]:
class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.inputs = dataframe["input"].tolist()
        self.targets = dataframe["target"].tolist()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {
            "input_text": self.inputs[idx],
            "target_text": self.targets[idx]
        }

train_dataset = TextDataset(train_data)
test_dataset = TextDataset(test_data)
val_dataset = TextDataset(val_data)

In [ ]:
def collate_fn(batch):
    input_texts = [item["input_text"] for item in batch]
    target_texts = [item["target_text"] for item in batch]

    # Tokenizar inputs
    input_encoding = tokenizer(
        input_texts,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Tokenizar targets
    with tokenizer.as_target_tokenizer():
        target_encoding = tokenizer(
            target_texts,
            max_length=512,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

    labels = target_encoding["input_ids"]
    labels[labels == tokenizer.pad_token_id] = -100

    return {
        "input_ids": input_encoding["input_ids"],
        "attention_mask": input_encoding["attention_mask"],
        "labels": target_encoding["input_ids"]
    }

train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

#**BILSTM-LSTM**

In [ ]:
# =============================
# 1. Tokenización
# =============================
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(df['input'] + df['target'])

input_seq = tokenizer.texts_to_sequences(df['input'])
target_seq = tokenizer.texts_to_sequences(df['target'])

MAX_LEN = max(max(len(s) for s in input_seq), max(len(s) for s in target_seq))

input_seq = pad_sequences(input_seq, maxlen=MAX_LEN, padding='post', truncating='post')
target_seq = pad_sequences(target_seq, maxlen=MAX_LEN, padding='post', truncating='post')

X_train, X_test, y_train, y_test = train_test_split(input_seq, target_seq, test_size=0.25, random_state=42)
vocab_size = len(tokenizer.word_index) + 1

In [ ]:
# =============================
# 2. Modelo Encoder-Decoder BiLSTM + LSTM
# =============================
embedding_dim = 256
lstm_units = 128

# Encoder
encoder_inputs = Input(shape=(MAX_LEN,))
enc_emb = Embedding(vocab_size, embedding_dim, mask_zero=True)(encoder_inputs)
encoder_lstm = Bidirectional(LSTM(lstm_units, return_state=True))
encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(enc_emb)
state_h = Concatenate()([forward_h, backward_h])
state_c = Concatenate()([forward_c, backward_c])

# Decoder
decoder_inputs = Input(shape=(MAX_LEN,))
dec_emb = Embedding(vocab_size, embedding_dim, mask_zero=True)(decoder_inputs)
decoder_lstm = LSTM(lstm_units*2, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])
decoder_dense = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# =============================
# 3. Entrenamiento con EarlyStopping
# =============================

y_train_exp = np.expand_dims(y_train, -1)
y_test_exp = np.expand_dims(y_test, -1)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.summary()

start_train = time.time()
model.fit([X_train, X_train], y_train_exp,
          validation_data=([X_test, X_test], y_test_exp),
          epochs=256, batch_size=32, callbacks=[early_stop])
end_train = time.time()
print(f"\nTiempo de entrenamiento: {end_train - start_train:.2f} segundos")

In [ ]:
# =============================
# 4. Diccionario para decodificación
# =============================
index_to_word = {v:k for k,v in tokenizer.word_index.items()}

@tf.function
def model_predict(seq_input, seq_decoder):
    return model([seq_input, seq_decoder], training=False)

def remove_consecutive_duplicates(words):
    if not words:
        return words
    filtered = [words[0]]
    for w in words[1:]:
        if w != filtered[-1]:
            filtered.append(w)
    return filtered

def predict_sentence(sentence):
    seq = tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred = model_predict(seq, seq)
    pred_ids = tf.argmax(pred, axis=-1).numpy()[0]
    pred_words = [index_to_word.get(idx, "") for idx in pred_ids if idx != 0]  # ignorar PAD
    pred_words = remove_consecutive_duplicates(pred_words)
    return [w for w in pred_words if w != ""]

In [ ]:
# =============================
# 5. Evaluación
# =============================
gleu_scores = []
bleu_scores = []
edit_scores = []
exact_match_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

start_eval = time.time()
print("\n--- Ejemplos y métricas ---")
for i in range(len(X_test)):
    input_text = ' '.join([index_to_word.get(idx, "") for idx in X_test[i] if idx != 0])
    expected = ' '.join([index_to_word.get(idx, "") for idx in y_test[i] if idx != 0])
    predicted = ' '.join(predict_sentence(input_text))

    ref_tokens = expected.split()
    gen_tokens = predicted.split()

    # Métricas
    gleu_scores.append(sentence_gleu([ref_tokens], gen_tokens))
    bleu_scores.append(sentence_bleu([ref_tokens], gen_tokens))
    edit_scores.append(edit_distance(ref_tokens, gen_tokens))
    exact_match_scores.append(int(ref_tokens == gen_tokens))

    # Precision, Recall, F1
    all_tokens = list(set(ref_tokens + gen_tokens))
    ref_bin = [1 if t in ref_tokens else 0 for t in all_tokens]
    pred_bin = [1 if t in gen_tokens else 0 for t in all_tokens]
    precision_scores.append(precision_score(ref_bin, pred_bin, zero_division=0))
    recall_scores.append(recall_score(ref_bin, pred_bin, zero_division=0))
    f1_scores.append(f1_score(ref_bin, pred_bin, zero_division=0))

    # Mostrar algunos ejemplos
    print(f"\nEntrada:   {input_text}")
    print(f"Esperado:  {expected}")
    print(f"Generado:  {predicted}")
    print(f"GLEU: {gleu_scores[-1]:.4f}, BLEU: {bleu_scores[-1]:.4f}, Edit Distance: {edit_scores[-1]}, Exact Match: {exact_match_scores[-1]}")

end_eval = time.time()
print(f"\nTiempo de evaluación: {end_eval - start_eval:.2f} segundos")

# Resultados promedio
print("\n--- Promedios ---")
print(f"GLEU: {np.mean(gleu_scores):.4f}")
print(f"BLEU: {np.mean(bleu_scores):.4f}")
print(f"Edit Distance: {np.mean(edit_scores):.2f}")
print(f"Exact Match: {np.mean(exact_match_scores):.4f}")
print(f"Precision: {np.mean(precision_scores):.4f}")
print(f"Recall: {np.mean(recall_scores):.4f}")
print(f"F1: {np.mean(f1_scores):.4f}")

#**mt5**

In [ ]:
# =============================
# 1. Carga del modelo y tokenizador
# =============================

tokenizer = MT5Tokenizer.from_pretrained('google/mt5-small')

# Detectar dispositivo inicial
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo inicial: {device}")

model = MT5ForConditionalGeneration.from_pretrained('google/mt5-small')
model.to(device)

In [ ]:
# =============================
# 2. Entrenamiento con EarlyStopping
# =============================

optimizer = AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# Parámetros de entrenamiento
num_epochs = 20
best_eval_loss = float('inf')
patience = 5  # Número de épocas sin mejora antes de detener
epochs_without_improvement = 0
model_save_path = "corrector_mT5_tu_impersonal_2_early_stopping"

# Entrenamiento con early stopping
print("Iniciando entrenamiento...")
training_start = time.perf_counter()
for epoch in range(num_epochs):
    try:
        # Entrenamiento
        model.train()
        total_train_loss = 0
        for batch in train_dataloader:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_train_loss += loss.item()

            loss.backward()
            optimizer.step()

        avg_train_loss = total_train_loss / len(train_dataloader)
        print(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}")

        # Evaluación (solo pérdida) para guardar el mejor modelo
        model.eval()
        total_eval_loss = 0
        with torch.no_grad():
            for batch in test_dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_eval_loss += outputs.loss.item()

        avg_eval_loss = total_eval_loss / len(test_dataloader)
        print(f"Epoch {epoch + 1}/{num_epochs}, Eval Loss: {avg_eval_loss:.4f}")

        # Guardar el mejor modelo y reiniciar paciencia
        if avg_eval_loss < best_eval_loss:
            best_eval_loss = avg_eval_loss
            model.save_pretrained(model_save_path)
            tokenizer.save_pretrained(model_save_path)
            print(f"Modelo guardado en {model_save_path} con Eval Loss: {best_eval_loss:.4f}")
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            print(f"No hay mejora. Épocas sin mejora: {epochs_without_improvement}/{patience}")

        # Detener si no hay mejora
        if epochs_without_improvement >= patience:
            print(f"\n⏹️ Early stopping activado. No hubo mejora en {patience} épocas consecutivas.")
            break

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Error: {e}. Cambiando a CPU y reiniciando época.")
            device = torch.device("cpu")
            model.to(device)
            torch.cuda.empty_cache()
            epoch -= 1
            continue
        else:
            raise e

training_end = time.perf_counter()
training_time = training_end - training_start
print(f"\n⏱️ Tiempo total de entrenamiento: {training_time:.2f} segundos")

In [ ]:
# =============================
# 3. Copia del modelo entrenado en drive
# =============================

local_model_path = "/content/corrector_mT5_tu_impersonal_2_early_stopping"
drive_model_path = "/content/drive/MyDrive/modelos_guardados_TFM/mT5_tu_impersonal"

shutil.copytree(local_model_path, drive_model_path, dirs_exist_ok=True)
print(f"Modelo copiado a: {drive_model_path}")

In [ ]:
# =============================
# 4. Evaluación
# =============================

model.eval()
total_eval_loss = 0
total_bleu = 0
total_gleu = 0
total_edit_distance = 0
num_exact_matches = 0
predictions = []
references = []
num_bleu = 0

with torch.no_grad():
    for batch_idx, batch in enumerate(test_dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_eval_loss += outputs.loss.item()

        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            min_length=10,
            num_beams=3,
            repetition_penalty=1.2,
            early_stopping=True
        )

        input_texts = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        generated_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        label_ids = labels.clone()
        label_ids[label_ids == -100] = tokenizer.pad_token_id
        target_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        for i in range(len(input_texts)):
            input_text = input_texts[i]
            generated_text = generated_texts[i]
            target_text = target_texts[i]

            # BLEU
            ref_tokens = target_text.split()
            gen_tokens = generated_text.split()
            max_n = min(len(ref_tokens), len(gen_tokens), 4)
            bleu = sentence_bleu([ref_tokens], gen_tokens, weights=[1/max_n]*max_n) if max_n > 0 else 0
            total_bleu += bleu
            num_bleu += 1

            # GLEU
            gleu = sentence_gleu([ref_tokens], gen_tokens)
            total_gleu += gleu

            # Edit Distance
            edit_dist = editdistance.eval(generated_text, target_text)
            total_edit_distance += edit_dist

            # Exact match
            if generated_text.strip() == target_text.strip():
                num_exact_matches += 1

            predictions.append(generated_text)
            references.append(target_text)

            print(f"\nEvaluación - Batch {batch_idx + 1}, Ejemplo {i + 1}:")
            print(f"Original: {input_text}")
            print(f"Generada: {generated_text}")
            print(f"Correcta: {target_text}")
            print(f"BLEU: {bleu:.4f}, GLEU: {gleu:.4f}, EditDist: {edit_dist}")

# Finales
avg_eval_loss = total_eval_loss / len(test_dataloader)
avg_bleu = total_bleu / num_bleu
avg_gleu = total_gleu / num_bleu
avg_edit_distance = total_edit_distance / num_bleu
exact_match_acc = num_exact_matches / num_bleu

print(f"\n--- Evaluación final ---")
print(f"Loss promedio: {avg_eval_loss:.4f}")
print(f"BLEU promedio: {avg_bleu:.4f}")
print(f"GLEU promedio: {avg_gleu:.4f}")
print(f"Edit Distance promedio: {avg_edit_distance:.2f}")
print(f"Exact Match Accuracy: {exact_match_acc:.4f}")
print(f"Mejor modelo guardado en: {model_save_path}")

evaluation_end = time.perf_counter()
evaluation_time = evaluation_end - evaluation_start
print(f"⏱️ Tiempo total de evaluación: {evaluation_time:.2f} segundos")

#**Mistral**

In [ ]:
!pip install -U bitsandbytes

In [ ]:
!huggingface-cli login

In [ ]:
# Definición de prompts (tú impersonal)

prompts = {
    "Prompt - Completo (system prompt, main prompt y ejemplos)": """
<s>[INST] <<SYS>>
Eres un asistente experto en corrección de textos en español, especializado en transformar el uso del pronombre "tú" con valor impersonal en formas impersonales correctas con "se" + verbo en tercera persona, ajustando tiempo verbal y concordancia. Corrige también otros errores ortográficos y gramaticales. El texto corregido debe tener un registro formal adecuado para trabajos académicos o científicos.

Devuelve únicamente el texto corregido, sin explicaciones ni el texto original.
<</SYS>>

Ejemplos de frases con errores:
1. Si no estudias, no apruebas los exámenes.
2. Cuando comes mucho, te duele el estómago.
3. En la investigación científica, si tú interpretas incorrectamente los resultados, puedes generar grandes incongruencias.

Ejemplos corregidos:
1. Si no se estudia, no se aprueban los exámenes.
2. Cuando se come mucho, duele el estómago.
3. En la investigación científica, si se interpretan incorrectamente los resultados, se pueden generar grandes incongruencias.

Ahora corrige este texto en español:
[DOCUMENTO]

Recuerda transformar cualquier uso del pronombre "tú" con valor impersonal en una forma impersonal adecuada con "se" + verbo en tercera persona, ajustando tiempo y concordancia. Devuelve solo el texto corregido en español, sin explicaciones ni el original.
[/INST]
"""
}

In [ ]:
# Definición de prompts (gerundio de posterioridad)

# prompts = {
#     "Prompt 4 - Completo (system prompt, main prompt y ejemplos)": """
# <s>[INST] <<SYS>>
# Eres un asistente experto en corrección de textos en español. Tu única tarea es identificar y corregir los casos de gerundio de posterioridad: cuando el gerundio indica una acción que ocurre después de la acción principal. En esos casos, debes transformar la construcción en una coordinación con "y" + verbo en forma personal, ajustando el tiempo verbal y la concordancia únicamente si es necesario. No cambies palabras, puntuación ni estructura. No agregues explicaciones ni contenido adicional. Mantén un registro formal académico o científico.

# Devuelve únicamente el texto corregido, sin explicaciones ni el texto original.
# <</SYS>>

# Ejemplos de frases con errores:
# 1. Se denunció la desaparición del joven, siendo hallado dos semanas después.
# 2. El investigador aplicó la encuesta, obteniendo los resultados al día siguiente.
# 3. El equipo realizó la prueba, descubriendo errores en el procedimiento.
# 4. El profesor explicó el tema, aclarando las dudas en la siguiente clase.

# Ejemplos corregidos:
# 1. Se denunció la desaparición del joven y fue hallado dos semanas después.
# 2. El investigador aplicó la encuesta y obtuvo los resultados al día siguiente.
# 3. El equipo realizó la prueba y descubrió errores en el procedimiento.
# 4. El profesor explicó el tema y aclaró las dudas en la siguiente clase.

# Ahora corrige este texto en español:
# [DOCUMENTO]

# Recuerda: solo transforma los gerundios de posterioridad, sin cambiar otra cosa. Devuelve únicamente el texto corregido.
# [/INST]
# """
# }

In [ ]:
# =============================
# 1. Configuración del dispositivo, cuantización y carga del modelo
# =============================

# Detectar dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Configuración de cuantización a 4 bits (BitsAndBytes)
nf4_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bfloat16
)

# Carga del modelo y tokenizador
model_id = 'mistralai/Mistral-7B-Instruct-v0.3'

tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=nf4_config,
    device_map='auto',
)
model.eval()

# Configuración de padding si no está definido
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = model.config.eos_token_id

# Creación de pipeline de generación
generator = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task='text-generation',
    device_map="auto",
    batch_size=15,
    torch_dtype="auto"
)

In [ ]:
# =============================
# 2. Evaluación
# =============================

resultados_por_prompt = {}
smoothie = SmoothingFunction().method4

full_test_data = df

for prompt_name, full_prompt in prompts.items():
    print(f"\n📊 Evaluando prompt: {prompt_name}")

    start_time = time.perf_counter()

    total_bleu = total_gleu = total_edit_distance = 0
    num_exact_matches = 0
    num_bleu = 0
    num_gleu = 0

    for i, (_, row) in enumerate(full_test_data.iterrows()):
        frase = row['input']
        target = row['target']
        prompt_final = full_prompt.replace("[DOCUMENTO]", frase)

        generado = generator(
            prompt_final,
            temperature = 0.35,
            repetition_penalty = 0.95,
            return_full_text=True
        )
        salida_modelo = generado[0]['generated_text'].split("[/INST]")[-1].strip()

        ref_tokens = target.split()
        gen_tokens = salida_modelo.split()

        # BLEU
        max_n = min(len(ref_tokens), len(gen_tokens), 4)
        if max_n == 0:
            bleu = 0
        else:
            weights = tuple([1.0 / max_n] * max_n)
            bleu = sentence_bleu([ref_tokens], gen_tokens, weights=weights, smoothing_function=smoothie)
            total_bleu += bleu
            num_bleu += 1

        # GLEU
        gleu = sentence_gleu([ref_tokens], gen_tokens)
        print(f"GLEU: ", {gleu})
        total_gleu += gleu
        num_gleu += 1

        # Edit Distance
        edit_dist = editdistance.eval(salida_modelo, target)
        total_edit_distance += edit_dist

        # Exact Match
        if salida_modelo.strip() == target.strip():
            num_exact_matches += 1

        # Progreso
        if (i + 1) % 10 == 0 or (i + 1) == len(full_test_data):
            print(f"Generación número: {i + 1}/{len(full_test_data)}")

    end_time = time.perf_counter()
    elapsed_time = end_time - start_time
    print(f"⏱️ Tiempo para evaluar '{prompt_name}': {elapsed_time:.2f} segundos")

    # Promedios
    avg_bleu = total_bleu / num_bleu if num_bleu > 0 else 0
    avg_gleu = total_gleu / num_gleu if num_gleu > 0 else 0
    avg_edit_distance = total_edit_distance / len(full_test_data)
    exact_match_acc = num_exact_matches / len(full_test_data)

    resultados_por_prompt[prompt_name] = {
        "BLEU": avg_bleu,
        "GLEU": avg_gleu,
        "Edit Distance": avg_edit_distance,
        "Exact Match": exact_match_acc,
    }

# Mostrar resultados finales
for prompt, resultados in resultados_por_prompt.items():
    print(f"\n--- Resultados para {prompt} ---")
    for metrica, valor in resultados.items():
        print(f"{metrica}: {valor:.4f}")